# NB_01 — Absorber Manufacturing Synthesis v2

This notebook reads completed engineering source records plus repository-level synthesis definitions:

```text
engineering_navigator/synthesis/engineering_concepts.yaml
engineering_navigator/synthesis/synthesis_rules.yaml
```

It derives:

- cross-source engineering concepts;
- candidate specifications;
- open specifications;
- the next engineering notebook.

The notebook contains the synthesis engine. Engineering concepts and specification rules live in YAML.


## 1. Configuration and repository paths

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import subprocess
import zipfile

import pandas as pd
import yaml

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE: str | Path | None = None

SOURCE_FILES = [
    "SOURCE_00_becker_transition_models.yaml",
    "SOURCE_01_bismuth_microstructure.yaml",
    "SOURCE_02_eliminating_nongaussian_spectral_response.yaml",
]
SYNTHESIS_ID = "SYNTHESIS_01"


def find_repo_root() -> Path:
    candidates = []
    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser().resolve())

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])
    candidates.extend(
        [
            Path("/content/sensors-becker"),
            Path("/home/dan/sensors-becker"),
            Path.home() / "sensors-becker",
        ]
    )

    for candidate in candidates:
        if candidate.is_dir() and (candidate / "engineering_navigator").is_dir():
            return candidate

    if Path("/content").exists():
        target = Path("/content/sensors-becker")
        if not target.exists():
            subprocess.run(["git", "clone", REPOSITORY_URL, str(target)], check=True)
        return target

    raise FileNotFoundError(
        "Could not locate sensors-becker. Set REPO_ROOT_OVERRIDE explicitly."
    )


REPO_ROOT = find_repo_root()
SOURCE_DIR = REPO_ROOT / "engineering_navigator" / "absorber_manufacturing" / "source_records"
SYNTHESIS_DIR = REPO_ROOT / "engineering_navigator" / "synthesis"
CONCEPTS_FILE = SYNTHESIS_DIR / "engineering_concepts.yaml"
RULES_FILE = SYNTHESIS_DIR / "synthesis_rules.yaml"
OUTPUT_DIR = REPO_ROOT / "outputs" / "engineering_questions" / "absorber_manufacturing" / SYNTHESIS_ID
EXPORT_DIR = REPO_ROOT / "exports" / SYNTHESIS_ID
EXPORT_ZIP = REPO_ROOT / "exports" / f"{SYNTHESIS_ID}_export.zip"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)

print(f"Repository : {REPO_ROOT}")
print(f"Sources    : {SOURCE_DIR.relative_to(REPO_ROOT)}")
print(f"Concepts   : {CONCEPTS_FILE.relative_to(REPO_ROOT)}")
print(f"Rules      : {RULES_FILE.relative_to(REPO_ROOT)}")


## 2. Load YAML inputs

In [ ]:
def load_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing YAML: {path}")
    data = yaml.safe_load(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise TypeError(f"{path.name}: expected one top-level mapping")
    return data


concept_config = load_yaml(CONCEPTS_FILE)
rule_config = load_yaml(RULES_FILE)

records = {}
source_slots = {}

for index, filename in enumerate(SOURCE_FILES):
    record = load_yaml(SOURCE_DIR / filename)
    source_id = record.get("source_id")
    if not source_id:
        raise KeyError(f"{filename}: missing source_id")
    if source_id in records:
        raise ValueError(f"Duplicate source_id: {source_id}")

    records[source_id] = record
    source_slots[f"SOURCE_{index:02d}"] = source_id

print(f"Loaded {len(records)} source records.")
print(f"Loaded {len(concept_config.get('concepts', []))} relationship concepts.")
print(f"Loaded {len(rule_config.get('candidate_specifications', []))} candidate-spec rules.")


## 3. Validate source-record status

In [ ]:
status_rows = []
for source_id, record in records.items():
    status_rows.append({
        "source_id": source_id,
        "title": record.get("title", ""),
        "record_status": record.get("record_status", ""),
        "extraction_status": record.get("extraction_status", ""),
        "reported_values": len(record.get("reported_values", [])),
        "relationships": len(record.get("engineering_relationships", [])),
    })

status_df = pd.DataFrame(status_rows).sort_values("source_id").reset_index(drop=True)

incomplete = status_df[
    ~status_df["extraction_status"].astype(str).str.startswith("complete")
]
if not incomplete.empty:
    raise ValueError(
        "All source records must be complete before synthesis:\n"
        + incomplete[["source_id", "extraction_status"]].to_string(index=False)
    )

print("Source-record validation: PASS")
status_df


## 4. Build engineering-axis matrix from YAML concepts

In [ ]:
def searchable_record_text(record: dict) -> str:
    chunks = []
    for field in (
        "design_variables",
        "engineering_relationships",
        "engineering_constraints",
        "future_questions",
        "unreported_variables",
    ):
        chunks.append(str(record.get(field, "")))
    return " ".join(chunks).lower()


axis_rows = []
for axis in concept_config.get("engineering_axes", []):
    row = {"engineering_axis": axis["id"]}
    support_count = 0

    aliases = set(axis.get("aliases", []))
    keywords = [str(k).lower() for k in axis.get("keywords", [])]

    for source_id, record in records.items():
        variable_ids = {
            item.get("id")
            for item in record.get("design_variables", [])
            if isinstance(item, dict) and item.get("id")
        }
        alias_hits = sorted(variable_ids.intersection(aliases))
        text = searchable_record_text(record)
        keyword_hits = sorted(k for k in keywords if k in text)

        evidence = alias_hits or keyword_hits
        row[source_id] = ", ".join(alias_hits or keyword_hits) if evidence else "—"
        support_count += int(bool(evidence))

    row["source_count"] = support_count
    axis_rows.append(row)

variable_matrix = (
    pd.DataFrame(axis_rows)
    .sort_values(["source_count", "engineering_axis"], ascending=[False, True])
    .reset_index(drop=True)
)
variable_matrix


## 5. Collect quantitative evidence

In [ ]:
value_rows = []
for source_id, record in records.items():
    for item in record.get("reported_values", []):
        if not isinstance(item, dict):
            continue
        value_rows.append({
            "source_id": source_id,
            "object": item.get("object"),
            "variable": item.get("variable"),
            "value": item.get("value"),
            "unit": item.get("unit"),
            "condition": item.get("condition"),
            "source_page": item.get("source_page"),
        })

values_df = pd.DataFrame(value_rows)

FOCUS_VARIABLES = {
    "Tc", "Bi_thickness", "C", "G",
    "SEM_grain_size", "diffraction_grain_size",
    "average_grain_size", "average_grain_radius",
    "quantum_efficiency", "residual_resistance_ratio",
    "cloud_size", "delta_E", "predicted_delta_E",
}

focus_values = (
    values_df[values_df["variable"].isin(FOCUS_VARIABLES)]
    .sort_values(["variable", "source_id", "object"])
    .reset_index(drop=True)
)
focus_values


## 6. Match source relationships to YAML concepts

In [ ]:
def relationship_text(item: dict) -> str:
    return (
        f"{item.get('relationship', '')} "
        f"{item.get('engineering_effect', '')}"
    ).lower()


def concept_matches(text: str, concept: dict) -> bool:
    groups = concept.get("required_keyword_groups", [])
    for group in groups:
        if not any(str(keyword).lower() in text for keyword in group):
            return False
    return True


relationship_rows = []
concepts = concept_config.get("concepts", [])

for source_id, record in records.items():
    for index, item in enumerate(record.get("engineering_relationships", [])):
        if not isinstance(item, dict):
            continue
        text = relationship_text(item)
        matched = [
            concept["id"]
            for concept in concepts
            if concept_matches(text, concept)
        ]
        relationship_rows.append({
            "source_id": source_id,
            "relationship_index": index,
            "relationship": item.get("relationship", ""),
            "engineering_effect": item.get("engineering_effect", ""),
            "source_pages": item.get("source_pages", []),
            "concepts": matched,
        })

source_relationships_df = pd.DataFrame(relationship_rows)

concept_rows = []
for concept in concepts:
    concept_id = concept["id"]
    supporting = source_relationships_df[
        source_relationships_df["concepts"].apply(
            lambda values: concept_id in values
        )
    ]
    sources = sorted(supporting["source_id"].unique().tolist())
    concept_rows.append({
        "concept": concept_id,
        "category": concept.get("category", ""),
        "source_count": len(sources),
        "sources": sources,
        "relationship_count": len(supporting),
        "status": (
            "supported_across_sources"
            if len(sources) >= 2
            else "single_source_support"
        ),
    })

relationships_df = (
    pd.DataFrame(concept_rows)
    .sort_values(["source_count", "concept"], ascending=[False, True])
    .reset_index(drop=True)
)
relationships_df


## 7. Generate candidate specifications from synthesis_rules.yaml

In [ ]:
concept_index = relationships_df.set_index("concept").to_dict("index")

candidate_specifications = []
spec_number = 1

for rule in rule_config.get("candidate_specifications", []):
    concept = rule["concept"]
    result = concept_index.get(concept, {})

    if result.get("source_count", 0) < int(rule.get("min_sources", 1)):
        continue

    candidate_specifications.append({
        "spec_id": f"SPEC_AM_{spec_number:02d}",
        "concept": concept,
        "specification": rule["specification"],
        "evidence": result.get("sources", []),
        "source_count": result.get("source_count", 0),
        "state": "source_supported_candidate",
        "next_validation": rule.get("next_validation", ""),
    })
    spec_number += 1

specifications_df = pd.DataFrame(candidate_specifications)
specifications_df


## 8. Generate open specifications from source-record gaps

In [ ]:
gap_rows = []
for source_id, record in records.items():
    for gap in record.get("unreported_variables", []):
        gap_rows.append({"source_id": source_id, "gap": str(gap)})

gaps_df = pd.DataFrame(gap_rows)

open_items = []
for rule in rule_config.get("open_specifications", []):
    if gaps_df.empty:
        continue

    keywords = [str(k).lower() for k in rule.get("keywords", [])]
    mask = gaps_df["gap"].str.lower().apply(
        lambda text: any(keyword in text for keyword in keywords)
    )
    matches = gaps_df[mask]
    if matches.empty:
        continue

    sources = sorted(matches["source_id"].unique().tolist())
    open_items.append({
        "concept": rule["concept"],
        "open_specification": rule["open_specification"],
        "why_open": "; ".join(sorted(matches["gap"].unique().tolist())),
        "gap_sources": sources,
        "source_count": len(sources),
        "next_measurement": rule.get("next_measurement", ""),
    })

open_specs_df = (
    pd.DataFrame(open_items)
    .sort_values(["source_count", "open_specification"], ascending=[False, True])
    .reset_index(drop=True)
)
open_specs_df


## 9. Select next engineering notebook from rules

In [ ]:
open_concepts = set(open_specs_df["concept"]) if not open_specs_df.empty else set()

next_notebook = None
for rule in rule_config.get("next_notebooks", []):
    if rule["open_concept"] not in open_concepts:
        continue

    prefixes = tuple(rule.get("input_prefixes", []))
    selected_inputs = [
        filename
        for filename in SOURCE_FILES
        if not prefixes or filename.startswith(prefixes)
    ]

    next_notebook = {
        "id": rule["id"],
        "engineering_question": rule["engineering_question"],
        "inputs": selected_inputs,
        "outputs": rule.get("outputs", []),
    }
    break

if next_notebook is None:
    next_notebook = {
        "id": "NB_02_ABSORBER_MANUFACTURING_REFINEMENT",
        "engineering_question": "Which unresolved absorber-manufacturing specification should be evaluated next?",
        "inputs": SOURCE_FILES,
        "outputs": ["ranked unresolved specifications", "next measurement plan"],
    }

next_notebook


## 10. Write synthesis outputs

In [ ]:
status_csv = OUTPUT_DIR / "source_status.csv"
variable_matrix_csv = OUTPUT_DIR / "variable_matrix.csv"
focus_values_csv = OUTPUT_DIR / "quantitative_evidence.csv"
source_relationships_csv = OUTPUT_DIR / "source_relationships.csv"
relationships_csv = OUTPUT_DIR / "synthesis_relationships.csv"
specifications_csv = OUTPUT_DIR / "candidate_specifications.csv"
open_specs_csv = OUTPUT_DIR / "open_specifications.csv"
synthesis_json = OUTPUT_DIR / "synthesis_summary.json"

status_df.to_csv(status_csv, index=False)
variable_matrix.to_csv(variable_matrix_csv, index=False)
focus_values.to_csv(focus_values_csv, index=False)
source_relationships_df.to_csv(source_relationships_csv, index=False)
relationships_df.to_csv(relationships_csv, index=False)
specifications_df.to_csv(specifications_csv, index=False)
open_specs_df.to_csv(open_specs_csv, index=False)

synthesis_summary = {
    "synthesis_id": SYNTHESIS_ID,
    "sources": sorted(records),
    "source_slots": source_slots,
    "relationship_concepts": relationships_df.to_dict("records"),
    "candidate_specifications": candidate_specifications,
    "open_specifications": open_items,
    "next_notebook": next_notebook,
}
synthesis_json.write_text(
    json.dumps(synthesis_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

written_files = {
    "source_status": status_csv,
    "variable_matrix": variable_matrix_csv,
    "quantitative_evidence": focus_values_csv,
    "source_relationships": source_relationships_csv,
    "synthesis_relationships": relationships_csv,
    "candidate_specifications": specifications_csv,
    "open_specifications": open_specs_csv,
    "synthesis_summary": synthesis_json,
}

for name, path in written_files.items():
    print(f"{name:26} {path.relative_to(REPO_ROOT)}")


## 11. Build and download export ZIP

In [ ]:
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for path in written_files.values():
    shutil.copy2(path, EXPORT_DIR / path.name)

if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EXPORT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print(f"Export package: {EXPORT_ZIP}")
print(f"Size: {EXPORT_ZIP.stat().st_size:,} bytes")

try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("Automatic download is available only in Google Colab.")


## 12. Handoff

Inspect:

```text
candidate_specifications.csv
open_specifications.csv
source_relationships.csv
synthesis_relationships.csv
synthesis_summary.json
```

The synthesis definitions now live outside the notebook in repository YAML, so future drivers can reuse the same engine.

*Admissible generalizations trail leading specifications.*
